In [51]:
import pymupdf4llm
import re
import os
from langchain_text_splitters import MarkdownHeaderTextSplitter, JSFrameworkTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
import json
from elasticsearch import Elasticsearch
from sentence_transformers import CrossEncoder
load_dotenv()

True

In [ ]:
file_path = "Attention.pdf"
markdown = pymupdf4llm.to_markdown(file_path)

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4")
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers_to_split_on)
markdown_chunks = splitter.split_text(markdown)

In [ ]:
def additional_recursive_chunker(markdown_chunks : list[Document], chunk_size = 500, chunk_overlap = 100):

    good_chunk = []
    chunk_count = 0
    recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, 
                                                             chunk_overlap = chunk_overlap, 
                                                             separators=["\n\n", "\n", ". ", " ", ""])
    id = 1
    for chunk in markdown_chunks:
        if(len(chunk.page_content) > chunk_size):
            meta_data = chunk.metadata
            chunks = recursive_text_splitter.split_documents([chunk])
            for i in chunks:
                i.metadata['id'] = id
                chunk_count = chunk_count + 1
                i.metadata["chunk_id"] = chunk_count
                good_chunk.append(i)
                id += 1
            chunk_count = 0
        else:
            chunk.metadata['id'] = id
            id += 1
            good_chunk.append(chunk)
    return good_chunk

In [ ]:
# this is the final chunk and metadata which is to be stores and embedded
final_chunks = additional_recursive_chunker(markdown_chunks)

In [ ]:
embedding_model = OllamaEmbeddings(model="llama3.2")

In [ ]:
# embed each of the query and then store them in the pgvector
vector_store = Chroma(
    collection_name= "my_docs",
    embedding_function=embedding_model,
    persist_directory="./vector_embeddings"
)

In [ ]:
ids = vector_store.add_documents(
    documents = final_chunks,)

In [ ]:
filter_condition = {}
search_kwargs = {}

In [ ]:
if len(filter_condition) == 0:
    search_kwargs = {
        'k' : 5,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
    }
else:
    search_kwargs = {
        'k' : 5,
        "fetch_k" : 10,
        "lambda_mult" : 0.7,
        "filter" : {
            **filter_condition
        }
    },

In [ ]:
search_kwargs

In [ ]:
retriver = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = search_kwargs
)

### Evaluation of the rag system without hybrid search and reranking

#### Evaluation LLMs

In [ ]:
from openai import OpenAI
from groq import Groq
from ragas.llms import llm_factory

OllamaClient = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434"
)

GroqClient = Groq(
    api_key=os.getenv("api_key")
)

GroqEvaluation = llm_factory(
    model="llama-3.3-70b-versatile",
    client=GroqClient
)

OllamaEvaluation = llm_factory(
    model="llama3.2",
    provider="openai",
    client=OllamaClient
)

#### RAG LLMs

In [ ]:
OllamaLLM = ChatOllama(model="llama3.2")
GroqLLM = ChatGroq(
    api_key=os.getenv("api_key"),
    model="llama-3.3-70b-versatile"
)

#### Metrics

In [ ]:
from ragas.metrics.collections import ( 
    ContextPrecision,
    ContextRecall, 
    ContextRelevance, 
    Faithfulness, 
    AnswerCorrectness, 
    AnswerRelevancy )
from ragas import evaluate
from datasets import Dataset

In [ ]:
# evaluation retriever
def recall_k(total_relevant_chunks, total_relevant_chunks_retirevied):
    # -> how many of relevant chunks were retreived out of all the relevant chunks available
    if total_relevant_chunks == 0:
        return 0.0
    recall = total_relevant_chunks_retirevied/total_relevant_chunks
    return recall

def precision_k(total_retreived_chunks, total_relevant_ones):
    # -> how many of the retrieved chunks were relevant chunks to the the question
    if total_retreived_chunks == 0:
        return 0.0
    precision = total_relevant_ones/total_retreived_chunks
    return precision

# evaluate the ranker (reranker or retriever's ranker)
def rr(rank_of_relevant_chunk):
    # -> the first relevant chunk out of all the retrieved chunk which are returned in order
    rr_value = 1/rank_of_relevant_chunk if rank_of_relevant_chunk != 0 else 0
    return rr_value

def ndcg(llm):
    # -> this metrics check whether the chunks are arranged in an descending order of chunks on 
    # the basis of relevancy
    pass

#### dataset for evaluation
the golden dataset will contain the dataset which will contain:
- user_query
- response
- retreived_contexts
- reference 

In [ ]:
golden_dataset = [
    {
        "user_input": "What is the Transformer architecture?",
        "reference": "The Transformer is a neural network architecture based entirely on attention mechanisms, replacing recurrence and convolutions.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What is self-attention and how does it work?",
        "reference": "Self-attention is an attention mechanism in which queries, keys, and values come from the same source, specifically the output of the previous layer. It allows different positions in a sequence to be related when computing its representation.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What are Query, Key, and Value in the attention mechanism?",
        "reference": "An attention function maps a query and a set of key-value pairs to an output. Queries and keys are used to determine attention scores, while values contribute to the resulting output.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "Why does the Transformer use multi-head attention?",
        "reference": "Multi-head attention allows the Transformer to jointly attend to information from different representation subspaces at different positions, enabling it to capture different types of relationships simultaneously.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What is positional encoding and why is it necessary?",
        "reference": "Positional encoding injects information about the relative or absolute positions of tokens into the model. It is necessary because the Transformer contains neither recurrence nor convolution and therefore needs positional information to understand token order.",
        "retrieved_contexts": [],
        "response": ""
    },
    {
        "user_input": "What is the difference between encoder-decoder attention and masked self-attention?",
        "reference": "Encoder-decoder attention allows every decoder position to attend to all positions in the input sequence through encoder outputs. In contrast, masked self-attention prevents decoder positions from attending to subsequent positions, ensuring that future tokens are not accessed.",
        "retrieved_contexts": [],
        "response": ""
    }
]

In [ ]:
relevant_chunks_dataset = [
    {
        "query": "What is the Transformer architecture?",
        "relevant_chunks": [
            {
                "id": 3,
                "section": "Abstract",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
"""
            }
        ]
    },
    {
        "query": "What is self-attention and how is it used in the Transformer?",
        "relevant_chunks": [
            {
                "id": 16,
                "section": "Background",
                "chunk_id": 3,
                "relevance_score": 3,
                "content": """
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations.
"""
            },
            {
                "id": 40,
                "section": "Applications of Attention in our Model",
                "chunk_id": 2,
                "relevance_score": 3,
                "content": """
The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
"""
            },
            {
                "id": 41,
                "section": "Applications of Attention in our Model",
                "chunk_id": 3,
                "relevance_score": 2,
                "content": """
Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.
"""
            }
        ]
    },
    {
        "query": "What is scaled dot-product attention?",
        "relevant_chunks": [
            {
                "id": 27,
                "section": "Attention",
                "chunk_id": 1,
                "relevance_score": 2,
                "content": """
An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.
"""
            },
            {
                "id": 28,
                "section": "Scaled Dot-Product Attention",
                "chunk_id": 0,
                "relevance_score": 2,
                "content": """
We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the queries with all keys.
"""
            },
            {
                "id": 29,
                "section": "Multi-Head Attention",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. The query is multiplied with all keys, divided by √dk, and a softmax function is applied to obtain the weights on the values.
"""
            }
        ]
    },
    {
        "query": "How does multi-head attention work in the Transformer?",
        "relevant_chunks": [
            {
                "id": 34,
                "section": "Multi-Head Attention",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions.
"""
            },
            {
                "id": 35,
                "section": "Multi-Head Attention",
                "chunk_id": 2,
                "relevance_score": 2,
                "content": """
These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2.
"""
            },
            {
                "id": 36,
                "section": "Multi-Head Attention",
                "chunk_id": 3,
                "relevance_score": 2,
                "content": """
Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this.
"""
            }
        ]
    },
    {
        "query": "What is positional encoding and why is it needed in the Transformer?",
        "relevant_chunks": [
            {
                "id": 46,
                "section": "Positional Encoding",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence.
"""
            },
            {
                "id": 48,
                "section": "Positional Encoding",
                "chunk_id": 3,
                "relevance_score": 2,
                "content": """
The positional encodings have the same dimension d_model as the embeddings, so that the two can be summed. There are many choices of positional encodings, learned and fixed.
"""
            },
            {
                "id": 50,
                "section": "Positional Encoding",
                "chunk_id": 5,
                "relevance_score": 2,
                "content": """
That is, each dimension of the positional encoding corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions.
"""
            }
        ]
    },
    {
        "query": "How is attention used in the encoder and decoder of the Transformer?",
        "relevant_chunks": [
            {
                "id": 39,
                "section": "Applications of Attention in our Model",
                "chunk_id": 1,
                "relevance_score": 3,
                "content": """
In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence.
"""
            },
            {
                "id": 40,
                "section": "Applications of Attention in our Model",
                "chunk_id": 2,
                "relevance_score": 3,
                "content": """
The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
"""
            },
            {
                "id": 41,
                "section": "Applications of Attention in our Model",
                "chunk_id": 3,
                "relevance_score": 3,
                "content": """
Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.
"""
            }
        ]
    }
]

In [ ]:
precision_recall_dataset = []
for question in relevant_chunks_dataset:
    question_dict = {}
    question_dict['query'] = question['query']
    question_dict['relevant_chunk_ids'] = [ chunk['id'] for chunk in question['relevant_chunks']]
    question_dict['relevancy_score'] = [ chunk['relevance_score'] for chunk in question['relevant_chunks']]
    precision_recall_dataset.append(question_dict)

precision_recall_dataset

In [ ]:
# retrieve the chunk from the vector store
retrieved_content = []
for key, question in enumerate(relevant_chunks_dataset):
    retrieved_chunks = retriver.invoke(question['query'])
    retrieved_content.append([chunk.page_content for chunk in retrieved_chunks])
    retrieved_chunks_ids = [ chunk.metadata['id'] for chunk in retrieved_chunks]
    precision_recall_dataset[key]['retrieved_chunk_ids'] = retrieved_chunks_ids

precision_recall_dataset

#### Precision and Recall

In [ ]:
total_precision = []
total_recall = []
for question in precision_recall_dataset:

    # total relevant among retreived
    relevant_among_retrieved = 0
    for i in retrieved_chunks_ids:
        if i in question['relevant_chunk_ids']:
            relevant_among_retrieved += 1

    # precision calculation
    precision_value = precision_k(5, relevant_among_retrieved)
    # recall calcualtion
    recall_value = recall_k(len(question["relevant_chunk_ids"]), relevant_among_retrieved)

    total_precision.append(precision_value)
    total_recall.append(recall_value)


average_precision = sum(total_precision)/5
average_recall = sum(total_recall)/5

print("Precision:", average_precision)
print("Recall:", average_recall)

#### MRR and nDGC

In [ ]:
# MRR
from functools import reduce
# calculate the rank for each of the query (RR):-
ranks = []
number_of_chunks = 5
for question in precision_recall_dataset:
    status = False
    for rank, retrieved_chunk_id in enumerate(question['retrieved_chunk_ids']):
        if retrieved_chunk_id in question['relevant_chunk_ids']:
            # print(rank+1, retrieved_chunk_id)
            # print(question['relevant_chunk_ids'])
            status = True
            ranks.append(1/(rank+1))
            break
    if(status == False):
        ranks.append(0)
print(ranks)

# calculating MRR now:-
mrr = reduce(lambda acc, current : acc + current, ranks, 0)/number_of_chunks
print("mean reciprocal rank =>", mrr)

In [ ]:
# NDCG
# it is used to check the overall ranking, not only the first chunks as mrr.
# it sees the chunk and then sees if the chunk is ranked above the less relevant or irrelevant chunk.
score = []
retrieved_chunk = []
for question in precision_recall_dataset:
    relevancy = []
    chunk = []
    for id in question['retrieved_chunk_ids']:
        chunk.append(id)
        if id in question['relevant_chunk_ids']:
            index = question['relevant_chunk_ids'].index(id)
            relevancy_chunk_score = question['relevancy_score'][index]
            relevancy.append(relevancy_chunk_score)
        else:
            relevancy.append(0)
    score.append(relevancy)
    retrieved_chunk.append(chunk)
print(score)
print(retrieved_chunk)

In [ ]:
import math

# calculate DCG:-
ndcg = []
number_of_chunks = 5
for s in score:
    actual_order = sorted(s, reverse=True)
    single_chunk_dcg = []
    single_chunk_idcg = []
    for rank in range(len(actual_order)):
        dcg_value = s[rank]/ math.log2(rank+2)
        idcg_value = actual_order[rank]/ math.log2(rank+2)
        single_chunk_dcg.append(dcg_value)
        single_chunk_idcg.append(idcg_value)
    dcg = (round(sum(single_chunk_dcg),3))
    idcg = (round(sum(single_chunk_idcg), 3))
    if idcg == 0:
        ndcg.append(0)
    else:
        ndcg.append(round(dcg/idcg, 3))
print(ndcg)

#### LLM based evaluation:-
- Context Recall
- Context Precision
- Context Relevancy
- Faithfullness
- Answer correctness
- Answer faithfulness

In [ ]:
real_dataset = {
    "user_input": [
        "What is the Transformer architecture?",
        "What is self-attention and how does it work?",
        "What are Query, Key, and Value in the attention mechanism?",
        "Why does the Transformer use multi-head attention?",
        "What is positional encoding and why is it necessary?",
        "What is the difference between encoder-decoder attention and masked self-attention?"
    ],

    "retrieved_contexts": [
        [],
        [],
        [],
        [],
        [],
        []
    ],

    "response": [
        "",
        "",
        "",
        "",
        "",
        ""
    ],

    "reference": [
        "The Transformer is a neural network architecture based entirely on attention mechanisms, replacing recurrence and convolutions.",

        "Self-attention is an attention mechanism in which queries, keys, and values come from the same source, specifically the output of the previous layer. It allows different positions in a sequence to be related when computing its representation.",

        "An attention function maps a query and a set of key-value pairs to an output. Queries and keys are used to determine attention scores, while values contribute to the resulting output.",

        "Multi-head attention allows the Transformer to jointly attend to information from different representation subspaces at different positions, enabling it to capture different types of relationships simultaneously.",

        "Positional encoding injects information about the relative or absolute positions of tokens into the model. It is necessary because the Transformer contains neither recurrence nor convolution and therefore needs positional information to understand token order.",

        "Encoder-decoder attention allows every decoder position to attend to all positions in the input sequence through encoder outputs. In contrast, masked self-attention prevents decoder positions from attending to subsequent positions, ensuring that future tokens are not accessed."
    ]
}

In [ ]:
real_dataset['retrieved_contexts'] = retrieved_content

In [ ]:
real_dataset


In [ ]:
# generating the asnwer for each question using llm:-
questions = real_dataset['user_input']
chunks = real_dataset['retrieved_contexts']
response = []
prompt = PromptTemplate(template = """You are a helpful AI assistant that answers questions using the provided context.

### Instructions:
1. Answer the user's question using only the information provided in the context.
2. Do not introduce information from your own knowledge.
3. If the context does not contain sufficient information to answer the question, explicitly state that you don't have enough information.
4. Provide a clear, accurate, and concise answer.
5. Do not invent facts or make unsupported assumptions.

### Context:
{context}

### Question:
{question}

### Answer:""", input_variables = ['context', 'question'])

for i in range(0, len(real_dataset['user_input'])):
    output = prompt | OllamaLLM
    answer = output.invoke({
        'context' : chunks[i],
        'question' : questions[i]
    }).content
    response.append(answer)

real_dataset['response'] = response

In [ ]:
real_dataset

In [ ]:
eval_dataset = Dataset.from_dict(real_dataset)
eval_dataset

In [ ]:
faithulness = Faithfulness(llm = OllamaEvaluation)
contextRecall = ContextRecall(llm = OllamaEvaluation)
contextPrecision = ContextPrecision(llm = OllamaEvaluation)
contextRelevance = ContextRelevance(llm = OllamaEvaluation)
answerCorrectness = AnswerCorrectness(llm = OllamaEvaluation, embeddings = embedding_model)
# answerRelevancy = AnswerRelevancy(llm = OllamaEvaluation, embeddings= embedding_model) 

In [ ]:
results = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithulness,
        contextRelevance,
        contextPrecision,
        contextRecall,
        answerCorrectness
    ]
)

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

### BM25 as the keyword search
#### Elastic Search will also perform bm25 algorithm inside of it

In [ ]:
# keyword search is done using the bm25 algorithm
els = Elasticsearch("http://localhost:9200")

In [ ]:
chunks_data = vector_store.get()

In [ ]:
if els.indices.exists(index="my_docs"):
    els.indices.delete(index="my_docs")
    print("index deleted")
else:
    print("index doesnot exist")

In [ ]:
for item in range(len(chunks_data['ids'])):
    els.index(
        index="my_docs",
        id = chunks_data['ids'][item],
        document={
            'chunk_id' : chunks_data['ids'][item],
            'content' : chunks_data['documents'][item]
        }
    )

In [ ]:
# extract result
def extract_result(query):
    response = els.search(
        index = "my_docs",
        query = {
            'match' : {
                'content' : query
            }
        },
        size = 5
    )
    contents = []
    chunk_ids = []
    for hit in response["hits"]["hits"]:
        content = hit['_source']['content']
        chunk_id = hit['_source']['chunk_id']
        contents.append(content)
        chunk_ids.append(chunk_id)
    return [contents, chunk_ids]


### Combining the chunks from dense and sparse search

##### It is the process where we have to fuse the ranks made by the dense and sparse search, so that we can make the reranking of the data on the basis of the llm or context aware model

In [ ]:
# rrf (reciprocal rank fusion):
# we have to collect the chunk_ids that we have got from the two retriever and then apply the formula to it
def rrf():
    scores = {}
    dense_retriever_chunk_ids = []
    query = "What is the Transformer architecture?"
    chunks = retriver.invoke(query)
    sparse_contents, sparse_content_ids = extract_result(query)
    for chunk in chunks:
        dense_retriever_chunk_ids.append(chunk.id)

    for i in range(len(dense_retriever_chunk_ids)):
        if dense_retriever_chunk_ids[i] not in scores:
            scores[dense_retriever_chunk_ids[i]] = [1/(60 + (i + 1))]
        elif dense_retriever_chunk_ids[i] in scores:
            scores[dense_retriever_chunk_ids[i]].append(1/(60 + (i + 1)))

    for i in range(len(sparse_content_ids)):
        if sparse_content_ids[i] not in scores:
            scores[sparse_content_ids[i]] = [1/(60 + (i + 1))]
        elif sparse_content_ids[i] in scores:
            scores[sparse_content_ids[i]].append(1/(60 + (i + 1)))
    print(scores)
    for i in scores.keys(): 
        scores[i] = sum(scores[i])
    return scores, sorted(scores.items(), key = lambda x : x[1], reverse= True)

In [52]:

scores, sorted_ids = rrf()

{'b7c78283-e27a-41b5-ae4e-c7d8f86dda27': [0.01639344262295082], '0ec7c641-09f6-4b53-b70d-6c0659e61b7e': [0.016129032258064516, 0.016129032258064516], '34454967-36e2-4547-8578-cfc2885273f0': [0.015873015873015872, 0.01639344262295082], '6cca5e8b-56c5-4cab-9387-3622a36adde1': [0.015625], '5fc7eaf2-9b44-4f06-93bc-33964a496588': [0.015384615384615385], '5d1ab307-2f67-4d2c-a368-30677c1a9cb8': [0.015873015873015872], '71691d34-ea76-4fe0-be12-f66d27412e14': [0.015625], 'b151c4c8-ffee-4263-9c99-9a980be09673': [0.015384615384615385]}


In [ ]:
sorted_ids

In [ ]:
chunks = []
for i in sorted_ids:
    chunks.append(vector_store.get_by_ids([i[0]])[0].page_content)
chunks

### Reranker

##### cross encoder reranker :=>
we will use the cross encoder as reranker becasue it will find the similarity of query and chunk together to find the score.

In [59]:
# cross encoder reranker
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 810.50it/s]


In [64]:
chunks

['The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.',
 'In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.',
 'In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention.',
 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best perfor

In [78]:
def rerank_document(query, documents, top_k = 5):
    # query and chunk pair
    pairs = [
        (query, chunk) for chunk in documents
    ]
    # similarity score
    points = reranker.predict(pairs)
    ranked = sorted(zip(documents, points), key = lambda x : x[1], reverse=True)
    return [
        content[0]
        for content in ranked
    ]

In [80]:
query = "What is the Transformer architecture?"
ranked_chunks = rerank_document(query, chunks)
ranked_chunks

['In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.',
 'The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.',
 'In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention.',
 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best perfor

### Generation of the answer from the llm

In [88]:
# generation
prompt_ = PromptTemplate(template="""You are the llm which is used in the rag, and you are given a list of context in the raned order, with the query.
                                     You have to produce the answer and if chunks dont contain the answer then, dont hallucinate and directly say that there is not context, 
                                     else answer the question on the basis of the chunks.
                                     Dont create a lengthy answer.
                                     Just give answer, like its q/a.
                                     query : {query} \n
                                     chunks : {chunks}\n
                                     answer : ----""",
                         input_variables= ['query', 'chunks'])
chain = prompt_ | OllamaLLM


for chunks in chain.stream({
    'query' : query,
    'chunks' : ranked_chunks
}):
    print(chunks.content, end = "", flush= True)


Q: What is the Transformer architecture?
A: The Transformer uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.

### Evaluate the chunks and answer after the changes in the pipeline

In [ ]:
# evaluation criterias